In [1]:
# =============================================================================
# GUS01F: Loading Numerical Data onto GeoTERYT Records
# =============================================================================
# This notebook demonstrates the v4.0 data storage capabilities:
# 1. Load BDL demographic data (subject P2137 - population)
# 2. Process and attach time series to TERYTRecord objects
# 3. Query data on individual records
# 4. Aggregate for regions (voivodeships)
# 5. Produce joint/marginal distributions
# 6. Save/reload database with data persistence
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

# Reload geoTERYT_db to get latest version (v4.0)
import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Data paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f"Repository root: {repo_root}")
print(f"Data root: {data_root}")
print(f"GUS root: {gus_root}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
Data root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS


In [2]:
# =============================================================================
# STEP 2: Load Complete GeoTERYT Database
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_geom_OW.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_geom_OW.pkl...
  Database version: 3.1
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4560 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
GeoTERYT Database Summary (v3.0)
Total records:           4,560
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes:  0
------------------------------------------------------------
Geometry:
  Records with geometr

In [3]:
# =============================================================================
# STEP 2B: Link Children to Parents (must happen before data loading)
# =============================================================================
db.link_children_to_parents()

Linking child units to their parents...
  ✓ Linked children to parents for 4560 records


In [4]:
# =============================================================================
# STEP 3: Load BDL Source Data
# =============================================================================
df_demographic = pd.read_csv(gus_root / "data" / 'bdl_demographic_data.csv', encoding='utf-8')
df_c_1988 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP1988_data.csv', encoding='utf-8')
df_c_2002 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2002_data.csv', encoding='utf-8')
df_c_2011 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2011_data.csv', encoding='utf-8')
df_c_2021 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2021_data.csv', encoding='utf-8')
df_c_2021_add = pd.read_csv(gus_root / "data" / "census_data" / 'P4315_Census_2021_educ_sex.csv',
                             encoding='utf-8', sep=';')

df_variables = pd.read_csv(gus_root / "metadata" / 'bdl_variables_level6.csv', encoding='utf-8')
df_c_variables = pd.read_csv(gus_root / "metadata" / 'census_meta.csv', encoding='utf-8')

print(f"df_demographic: {df_demographic.shape}")
print(f"df_variables: {df_variables.shape}")
print(f"df_c_2021_add: {df_c_2021_add.shape}")
print(f"  Columns: {list(df_c_2021_add.columns[:5])} ...")
print(f"\nAvailable subjects: {sorted(df_demographic['subjectId'].unique())}")

df_demographic: (431358, 5)
df_variables: (27922, 10)
df_c_2021_add: (4196, 33)
  Columns: ['Kod', 'Nazwa', 'ogółem;ogółem;2021;[osoba]', 'ogółem;wyższe;2021;[osoba]', 'ogółem;średnie i policealne - ogółem;2021;[osoba]'] ...

Available subjects: ['P1336', 'P2137', 'P2914']


In [5]:
# Edit years for census of 1988

for id, row in df_c_1988.iterrows():
    df_c_1988.at[id, 'values']= df_c_1988.at[id, 'values'].replace(", 'year': '1998'", ", 'year': '1988'")
    
df_c_variables['years'] = df_c_variables['years'].str.replace("[1998]", "[1988]")

# Unify the meta dataframe structure
df_c_variables['n4'] = None
df_c_variables['n5'] = None


In [6]:
# Collect all subject ids
# Subjects to ignore entirely (urban/rural splits, city-only data)
IGNORED_SUBJECTS = {'P4345', 'P3310', 'P1336', 'P2914'}

subject_ids = {"BDL": [], "Census": {"1988" : [], "2002": [], "2011": [], "2021": []}}

subject_ids["BDL"] = [s for s in df_demographic['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids["Census"]["1988"] = [s for s in df_c_1988['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids["Census"]["2002"] = [s for s in df_c_2002['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids["Census"]["2011"] = [s for s in df_c_2011['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids["Census"]["2021"] = [s for s in df_c_2021['subjectId'].unique() if s not in IGNORED_SUBJECTS]

subject_ids_flat = subject_ids['BDL'] + subject_ids["Census"]["1988"] + subject_ids["Census"]["2002"] \
    + subject_ids["Census"]["2011"] + subject_ids["Census"]["2021"]

subject_names_dict = {}
for subject in subject_ids_flat:
    subject_names_dict[subject] = ''
    
subject_ids

{'BDL': ['P2137'],
 'Census': {'1988': ['P2884', 'P2885', 'P2883', 'P2887'],
  '2002': ['P2114', 'P2403', 'P2402', 'P2871'],
  '2011': ['P3304', 'P3311', 'P3309', 'P3420'],
  '2021': ['P4253', 'P4320', 'P4287']}}

In [7]:
# Names of different subjects
# BDL subject
subject_names_dict['P2137'] = 'pop__age_sex'

# Census 1988
subject_names_dict['P2884'] = 'pop__age'
subject_names_dict['P2885'] = 'pop__educ'
subject_names_dict['P2883'] = 'pop__sex'
subject_names_dict['P2887'] = 'hh_size'

# Census 2002
subject_names_dict['P2114'] = 'pop__age_sex'
subject_names_dict['P2403'] = 'pop__age_educ'
subject_names_dict['P2402'] = 'pop__sex_educ'
subject_names_dict['P2871'] = 'hh_size'

# Census 2011
subject_names_dict['P3304'] = 'pop__age_sex'
subject_names_dict['P3311'] = 'pop__age_educ'
subject_names_dict['P3309'] = 'pop__sex_educ'
subject_names_dict['P3420'] = 'hh_size'

# Census 2021
subject_names_dict['P4253'] = 'pop__age_sex'
subject_names_dict['P4320'] = 'pop__age_educ'
subject_names_dict['P4315'] = 'pop__sex_educ'  # Wide-format CSV (added manually)
subject_names_dict['P4287'] = 'hh_size'

subject_names_dict

{'P2137': 'pop__age_sex',
 'P2884': 'pop__age',
 'P2885': 'pop__educ',
 'P2883': 'pop__sex',
 'P2887': 'hh_size',
 'P2114': 'pop__age_sex',
 'P2403': 'pop__age_educ',
 'P2402': 'pop__sex_educ',
 'P2871': 'hh_size',
 'P3304': 'pop__age_sex',
 'P3311': 'pop__age_educ',
 'P3309': 'pop__sex_educ',
 'P3420': 'hh_size',
 'P4253': 'pop__age_sex',
 'P4320': 'pop__age_educ',
 'P4287': 'hh_size',
 'P4315': 'pop__sex_educ'}

In [8]:
# =============================================================================
# STEP 4: Process Subjects
# =============================================================================
# Use the new process_subject_data() static method on GeoTERYTDatabase

# Example: Process subject P2137 (Population Data) from BDL
subject_id = 'P2137'
df_p2137 = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, subject_id)

print(f"Processed P2137: {df_p2137.shape}")
print(f"\nColumns: {list(df_p2137.columns)}")
print(f"\nCategory columns present:")
for col in ['n1', 'n2', 'n3', 'n4', 'n5']:
    if col in df_p2137.columns:
        print(f"  {col}: {sorted(df_p2137[col].dropna().unique())}")
print(f"\nYears: {sorted(df_p2137['year'].dropna().astype(str).unique())}")
print(f"Unique TERYT IDs: {df_p2137['teryt_id'].nunique()}")
df_p2137.head()

Processed P2137: (7355712, 11)

Columns: ['nuts_id', 'name', 'variableId', 'subjectId', 'var_id', 'n1', 'n2', 'year', 'val', 'attrId', 'teryt_id']

Category columns present:
  n1: ['0-14', '0-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70 i więcej', '70-74', '75-79', '80-84', '85 i więcej', 'ogółem']
  n2: ['kobiety', 'mężczyźni', 'ogółem']

Years: ['1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Unique TERYT IDs: 4556


,nuts_id,name,variableId,subjectId,var_id,n1,n2,year,val,attrId,teryt_id
0,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1995,38609399,1,0000000
1,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1996,38639341,1,0000000
2,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1997,38659979,1,0000000
3,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1998,38666983,1,0000000
4,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1999,38263303,1,0000000


In [9]:
# =============================================================================
# STEP 4: Process ALL Subjects
# =============================================================================

df_subjects = {
    "BDL": df_demographic,
    "Census": {
        "1988": df_c_1988,
        "2002": df_c_2002,
        "2011": df_c_2011,
        "2021": df_c_2021
    }
}

df_processed_subjects = {
    "BDL": {},
    "Census": {
        "1988": {},
        "2002": {},
        "2011": {},
        "2021": {}
    }
}

# --- Process BDL and Census subjects ---
for subject in subject_ids.items():
    if subject[0] == "BDL":
        subjects = subject[1]
        for s in subjects:
            print(f"Processing BDL subject: {s}...")
            df = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, s)
            df_processed_subjects["BDL"][s] = df
    else:
        for sub in subject[1].items():
            print(f"Processing Census subject: {sub[0]} - {sub[1]}...")
            year = sub[0]
            df_c = df_subjects["Census"][year]
            for s in sub[1]:
                print(f"  Processing subject: {s}...")
                df = gtdb.GeoTERYTDatabase.process_subject_data(df_c, df_c_variables, s)
                df_processed_subjects["Census"][year][s] = df

# --- P4315: Convert wide-format Census 2021 pop__sex_educ to long format ---
print("\nProcessing P4315 (wide-format Census 2021 pop__sex_educ)...")

rows = []
var_id_counter = 900001
var_id_lookup = {}  # (sex, educ) -> var_id

for col in df_c_2021_add.columns:
    if col in ['Kod', 'Nazwa']:
        continue
    parts = col.split(';')
    if len(parts) < 3:
        continue
    sex = parts[0].strip()
    educ = parts[1].strip()
    year = int(parts[2].strip())
    key = (sex, educ)
    if key not in var_id_lookup:
        var_id_lookup[key] = var_id_counter
        var_id_counter += 1
    vid = var_id_lookup[key]
    
    for _, row in df_c_2021_add.iterrows():
        kod = str(row['Kod']).zfill(7)
        val = row[col]
        if pd.isna(val):
            continue
        rows.append({
            'nuts_id': kod.ljust(12, '0'),
            'name': row['Nazwa'],
            'variableId': vid,
            'subjectId': 'P4315',
            'var_id': vid,
            'n1': sex,
            'n2': educ,
            'year': year,
            'val': float(val),
            'teryt_id': kod
        })

df_p4315 = pd.DataFrame(rows)
df_processed_subjects["Census"]["2021"]["P4315"] = df_p4315
# Also add to subject_ids so downstream loading picks it up
if 'P4315' not in subject_ids["Census"]["2021"]:
    subject_ids["Census"]["2021"].append('P4315')
print(f"  P4315: {df_p4315.shape[0]:,} rows, {df_p4315['teryt_id'].nunique()} units")
print(f"  Variables: {len(var_id_lookup)} (sex x education combos)")

# --- Fix hh_size subjects ---

# P3420 (Census 2011): Remove 'wskaźnik precyzji' rows, keep 'wartość liczbowa'.
# Remove 'ludność w gospodarstwach domowych' and 'przeciętna liczba osób'.
# Relabel to standard hh_size format for bin parsing compatibility.
# NOTE: verified that values ARE household counts (sum of bins = ogółem).
# NOTE: P3420 is powiat-level (11-digit IDs), not gmina-level.
if 'P3420' in df_processed_subjects["Census"]["2011"]:
    df = df_processed_subjects["Census"]["2011"]["P3420"]
    before = len(df)
    if 'n2' in df.columns:
        df = df[df['n2'] == 'wartość liczbowa'].copy()
        if df['n2'].nunique() <= 1:
            df = df.drop(columns=['n2'])
    if 'n1' in df.columns:
        exclude = ['ludność w gospodarstwach domowych',
                    'przeciętna liczba osób w gospodarstwie domowym']
        df = df[~df['n1'].isin(exclude)].copy()
        # Relabel to standard format for _parse_numeric_bounds compatibility
        label_map = {
            'osoby w gospodarstwie domowym - 1': '1-osobowe',
            'osoby w gospodarstwie domowym - 2': '2-osobowe',
            'osoby w gospodarstwie domowym - 3': '3-osobowe',
            'osoby w gospodarstwie domowym - 4': '4-osobowe',
            'osoby w gospodarstwie domowym - 5 i więcej': '5-osobowe i większe',
        }
        df['n1'] = df['n1'].replace(label_map)
    df_processed_subjects["Census"]["2011"]["P3420"] = df
    print(f"\n  P3420 fix: {before} -> {len(df)} rows (filtered + relabeled)")

# P2871 (Census 2002): Keep only 'gospodarstwa' rows (not 'ludność w gospodarstwach')
if 'P2871' in df_processed_subjects["Census"]["2002"]:
    df = df_processed_subjects["Census"]["2002"]["P2871"]
    before = len(df)
    if 'n1' in df.columns:
        df = df[df['n1'] == 'gospodarstwa'].copy()
        if df['n1'].nunique() <= 1:
            df = df.drop(columns=['n1'])
    df_processed_subjects["Census"]["2002"]["P2871"] = df
    print(f"  P2871 fix: {before} -> {len(df)} rows (kept only 'gospodarstwa')")

# P4287 (Census 2021): Remove 'ludność w gospodarstwach domowych' and 'przeciętna'
if 'P4287' in df_processed_subjects["Census"]["2021"]:
    df = df_processed_subjects["Census"]["2021"]["P4287"]
    before = len(df)
    if 'n1' in df.columns:
        exclude = ['ludność w gospodarstwach domowych',
                    'przeciętna liczba osób w gospodarstwie domowym']
        df = df[~df['n1'].isin(exclude)].copy()
    df_processed_subjects["Census"]["2021"]["P4287"] = df
    print(f"  P4287 fix: {before} -> {len(df)} rows (removed useless vars)")

del df_subjects
gc.collect()

# Save df_processed_subjects for later use
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'wb') as f:
    pickle.dump(df_processed_subjects, f)

Processing BDL subject: P2137...
Processing Census subject: 1988 - ['P2884', 'P2885', 'P2883', 'P2887']...
  Processing subject: P2884...
  Processing subject: P2885...
  Processing subject: P2883...
  Processing subject: P2887...
Processing Census subject: 2002 - ['P2114', 'P2403', 'P2402', 'P2871']...
  Processing subject: P2114...
  Processing subject: P2403...
  Processing subject: P2402...
  Processing subject: P2871...
Processing Census subject: 2011 - ['P3304', 'P3311', 'P3309', 'P3420']...
  Processing subject: P3304...
  Processing subject: P3311...
  Processing subject: P3309...
  Processing subject: P3420...
Processing Census subject: 2021 - ['P4253', 'P4320', 'P4287']...
  Processing subject: P4253...
  Processing subject: P4320...
  Processing subject: P4287...

Processing P4315 (wide-format Census 2021 pop__sex_educ)...
  P4315: 125,880 rows, 4196 units
  Variables: 30 (sex x education combos)

  P3420 fix: 6064 -> 2274 rows (filtered + relabeled)
  P2871 fix: 43776 -> 21

In [10]:
# Open the saved processed data to verify
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'rb') as f:
    df_processed_subjects = pickle.load(f)
    

In [11]:
# =============================================================================
# STEP 5: Load Subject Data onto TERYTRecords
# =============================================================================
# This attaches time series data to each matching TERYTRecord in the database

for subject in subject_ids.items():
    if subject[0] == "BDL":
        subjects = subject[1]
        for s in subjects:
            print(f"Loading BDL subject: {s}...")
            df = df_processed_subjects["BDL"][s]
            stats = db.load_subject_data(df, source_type='BDL', subject_id=s, subject_name=subject_names_dict[s])
            print(f"  Loading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
            print(f"  Total data points loaded: {stats['total_data_points']:,}")
    else:
        for sub in subject[1].items():
            print(f"Loading Census subject: {sub[0]} - {sub[1]}...")
            year = sub[0]
            for s in sub[1]:
                print(f"  Loading subject: {s}...")
                df = df_processed_subjects["Census"][year][s]
                stats = db.load_subject_data(df, source_type='Census', subject_id=s, subject_name=subject_names_dict[s])
                print(f"    Loading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
                print(f"    Total data points loaded: {stats['total_data_points']:,}")

# Free memory
del df_processed_subjects
gc.collect()

Loading BDL subject: P2137...
  ✓ Loaded 7,352,352 data points for subject P2137
  ✓ Matched 4549 TERYT records, 7 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0216001', '0410001', '1210001', '1431981', '1431991', '1465158', '1465998']
  Loading statistics: 4549 matched, 7 unmatched
  Total data points loaded: 7,352,352
Loading Census subject: 1988 - ['P2884', 'P2885', 'P2883', 'P2887']...
  Loading subject: P2884...
  ✓ Loaded 28,992 data points for subject P2884
  ✓ Matched 3624 TERYT records, 5 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0216001', '0410001', '1210001', '1431981', '1431991']
    Loading statistics: 3624 matched, 5 unmatched
    Total data points loaded: 28,992
  Loading subject: P2885...
  ✓ Loaded 14,496 data points for subject P2885
  ✓ Matched 3624 TERYT records, 5 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0216001', '0410001', '1210001', '1431981', '1431991']
    Loading statistics: 3624 matched, 5 unmatched
    Total data points loaded: 14,496
  Load

0

In [12]:
# =============================================================================
# STEP 6: Create Merged Subjects (unified census + BDL time series)
# =============================================================================
# Creates NEW merged subjects (M_ prefix) from groups sharing the same topic.
# Original subjects are NOT modified - raw data stays intact.
# For age dimensions: computes unified bins via common break points.
# For sex dimensions: exact label matching.

importlib.reload(gtdb)

print("Creating merged subjects...")
merged_info = db.create_merged_subjects(subject_names_dict)

# Update subject_names_dict with merged subjects
for merged_sid, source_ids in merged_info.items():
    group_name = merged_sid.replace('M_', '')
    subject_names_dict[merged_sid] = group_name
    print(f"  Added {merged_sid} -> '{group_name}'")

# Show data summary after merge
summary = db.get_data_summary()
print(f"\nAfter merge:")
print(f"  Records with data: {summary['records_with_data']}")
print(f"  Subjects: {summary['n_subjects']} ({summary['subjects']})")
print(f"  Total data series: {summary['total_data_series']:,}")
print(f"  Total data points: {summary['total_data_points']:,}")

Creating merged subjects...
Found 4 subject groups to merge:
  pop__age_sex: ['P2137', 'P2114', 'P3304', 'P4253']
  hh_size: ['P2887', 'P2871', 'P3420', 'P4287']
  pop__age_educ: ['P2403', 'P3311', 'P4320']
  pop__sex_educ: ['P2402', 'P3309', 'P4315']

  Merged subject: M_pop__age_sex
    n1 (['age']): 19 labels
    n2 (['sex']): 3 labels
    Aggregates detected in P2137 dim n1: {'70 i więcej', '0-14'}
    ✓ 4533 records, 257445 merged series created

  Merged subject: M_hh_size
    n1 (['age']): 5 labels
    ✓ 3624 records, 14496 merged series created

  Merged subject: M_pop__age_educ
    n1 (['age']): 11 labels
    n2 (['education']): 17 labels
    ✓ 381 records, 71060 merged series created

  Merged subject: M_pop__sex_educ
    n1 (['sex']): 3 labels
    n2 (['education']): 16 labels
    ✓ 4275 records, 193113 merged series created
  Added M_pop__age_sex -> 'pop__age_sex'
  Added M_hh_size -> 'hh_size'
  Added M_pop__age_educ -> 'pop__age_educ'
  Added M_pop__sex_educ -> 'pop__sex_

In [13]:
# =============================================================================
# STEP 7: Extract Total Population & Classify Urban/Rural
# =============================================================================

print("Extracting total population...")
n_pop = db.extract_population(subject_names_dict)

print("\nClassifying urban/rural...")
n_class = db.classify_population()

Extracting total population...
  ✓ Extracted population for 4533 records

Classifying urban/rural...
  ✓ Classified 3411 records by urban/rural


In [14]:
# =============================================================================
# STEP 8: Code Dimension Labels
# =============================================================================

print("Coding dimension labels...")
n_coded = db.code_dimension_labels(subject_names_dict)

Coding dimension labels...
  ✓ Coded dimension labels for 1938703 DataSeries across 21 subjects


In [15]:
# =============================================================================
# STEP 9: Save Database with All Data
# =============================================================================

save_path = geo_root / 'geoteryt_complete_final.pkl'
db.save_complete(save_path)

Saving complete database to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  ✓ Saved 4561 records
  ✓ Records with data: 4533
  ✓ File size: 1582.8 MB
  ✓ Path: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl


In [16]:
# =============================================================================
# STEP 10: Verify Data on Individual Records
# =============================================================================
data_summary = db.get_data_summary()
print("Data Summary:")
for k, v in data_summary.items():
    print(f"  {k}: {v}")

# Show subjects and their types
print("\nSubjects:")
for sid in sorted(data_summary['subjects']):
    sname = subject_names_dict.get(sid, '')
    prefix = "MERGED" if sid.startswith('M_') else "RAW"
    print(f"  [{prefix}] {sid}: {sname}")

Data Summary:
  records_with_data: 4533
  total_records: 4561
  subjects: ['M_hh_size', 'M_pop__age_educ', 'M_pop__age_sex', 'M_pop__sex_educ', 'P2114', 'P2137', 'P2402', 'P2403', 'P2871', 'P2883', 'P2884', 'P2885', 'P2887', 'P3304', 'P3309', 'P3311', 'P3420', 'P4253', 'P4287', 'P4315', 'P4320']
  n_subjects: 21
  total_data_series: 1938703
  total_data_points: 15310657

Subjects:
  [MERGED] M_hh_size: hh_size
  [MERGED] M_pop__age_educ: pop__age_educ
  [MERGED] M_pop__age_sex: pop__age_sex
  [MERGED] M_pop__sex_educ: pop__sex_educ
  [RAW] P2114: pop__age_sex
  [RAW] P2137: pop__age_sex
  [RAW] P2402: pop__sex_educ
  [RAW] P2403: pop__age_educ
  [RAW] P2871: hh_size
  [RAW] P2883: pop__sex
  [RAW] P2884: pop__age
  [RAW] P2885: pop__educ
  [RAW] P2887: hh_size
  [RAW] P3304: pop__age_sex
  [RAW] P3309: pop__sex_educ
  [RAW] P3311: pop__age_educ
  [RAW] P3420: hh_size
  [RAW] P4253: pop__age_sex
  [RAW] P4287: hh_size
  [RAW] P4315: pop__sex_educ
  [RAW] P4320: pop__age_educ
